
---
## Section 4 - Algorithm Selection & Mathematical Justification

We employ four models: one uninformative baseline and three substantive classifiers. This design mirrors the rubric requirement of identifying a "performance floor" and comparing algorithms of increasing sophistication.

### 4.1 M0 - Majority Class Baseline (Performance Floor)

**Purpose:** Establishes a trivial lower bound that any meaningful model must surpass. The baseline always predicts the majority class (Team A wins, since $P(y=1) \approx 0.525$).

$$
\hat{y} = \underset{c \in \{0,1\}}{\arg\max} \; P(Y = c \mid \text{training data}) = 1
$$

Any model with F1 $\leq$ F1$_{\text{baseline}}$ provides no predictive value over guessing.

---

### 4.2 M1 - Logistic Regression (Interpretable Linear Model)

**Mathematical Model:** Logistic Regression models the posterior probability of Team A winning as a sigmoid-transformed linear combination of features:

$$
P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^{\top} \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^{\top} \mathbf{x} + b)}}
$$

**Training Objective** - L2-regularised log-loss:

$$
\mathcal{L}_{LR}(\mathbf{w}, b) = -\frac{1}{n}\sum_{i=1}^{n} \bigl[y^{(i)} \log \hat{p}^{(i)} + (1-y^{(i)}) \log(1-\hat{p}^{(i)})\bigr] + \frac{\lambda}{2} \lVert \mathbf{w} \rVert_2^2
$$

where $\lambda = 1/C$ is the regularisation strength (we use $C = 1.0$, a balanced prior). The L2 term $\frac{\lambda}{2}\lVert\mathbf{w}\rVert_2^2$ penalises large weights, preventing overfitting to any single feature.

**Justification:** Logistic Regression serves as the **interpretable baseline**. The learned weights $\mathbf{w}$ directly quantify each feature's marginal contribution to win probability - invaluable for coaches and analysts who need to explain predictions to non-technical stakeholders.

---

### 4.3 M2 - Random Forest (Non-linear Ensemble)

**Mathematical Model:** Random Forest builds an ensemble of $T$ decision trees $\{h_t\}_{t=1}^T$, each trained on a bootstrap sample $\mathcal{D}_t$ of the training data and a random feature subset $\mathcal{F}_t \subseteq \{1,\ldots,p\}$ of size $\sqrt{p}$ at each split:

$$
\hat{p}_{RF}(\mathbf{x}) = \frac{1}{T} \sum_{t=1}^{T} h_t(\mathbf{x})
$$

Each split in tree $t$ maximises the reduction in **Gini impurity**:

$$
\Delta \text{Gini}(s, \mathcal{D}) = \text{Gini}(\mathcal{D}) - \frac{|\mathcal{D}_L|}{|\mathcal{D}|}\text{Gini}(\mathcal{D}_L) - \frac{|\mathcal{D}_R|}{|\mathcal{D}|}\text{Gini}(\mathcal{D}_R)
$$

where $\text{Gini}(\mathcal{D}) = 1 - \sum_{c} \hat{p}_c^2$.

**Justification:** Random Forest captures **non-linear interactions** between features - e.g., a high win-rate differential may matter more in high-stakes matches ($x_{10}$). The double randomisation (bagging + feature subsampling) provides strong regularisation against overfitting. Feature importance scores offer interpretability alongside predictive power.

---

### 4.4 M3 - XGBoost (Gradient Boosted Trees - State of the Art)

**Mathematical Model:** XGBoost builds an additive ensemble of $T$ trees sequentially. At step $t$, it adds the tree $f_t$ that minimises the second-order Taylor approximation of the loss:

$$
\mathcal{L}^{(t)} \approx \sum_{i=1}^{n} \left[ g_i f_t(\mathbf{x}^{(i)}) + \frac{1}{2} h_i f_t(\mathbf{x}^{(i)})^2 \right] + \Omega(f_t)
$$

where:
- $g_i = \partial_{\hat{p}^{(i)}} \mathcal{L}(y^{(i)}, \hat{p}^{(i)})$ is the first-order gradient of the loss
- $h_i = \partial^2_{\hat{p}^{(i)}} \mathcal{L}(y^{(i)}, \hat{p}^{(i)})$ is the Hessian (second-order curvature)
- $\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \lVert \mathbf{w} \rVert^2$ is the regularisation term (tree complexity penalty + L2 leaf weight penalty)

The final ensemble prediction is:

$$
\hat{p}_{XGB}(\mathbf{x}) = \sigma\!\left(\sum_{t=1}^{T} \eta \cdot f_t(\mathbf{x})\right)
$$

where $\eta$ is the learning rate (shrinkage), which scales each new tree's contribution to prevent over-correction.

**Justification:** XGBoost represents the **state-of-the-art** for tabular binary classification. The second-order Taylor expansion provides more precise gradient descent than first-order methods. Built-in L1/L2 regularisation, column subsampling, and learning rate shrinkage jointly combat overfitting - critical given our temporal distribution shift between training (2023–2024) and test (2025) seasons. The `colsample_bytree=0.8` parameter additionally reduces feature correlation between trees, improving ensemble diversity.
